In [6]:
#!/usr/bin/env python
# coding: utf-8
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from stargazer.stargazer import Stargazer
from datetime import timedelta, datetime
from IPython.display import display, Latex, Math, HTML
from linearmodels.panel import PanelOLS
import addfips
import warnings
import os

# Suppress warnings
warnings.filterwarnings('ignore')

#Read in data files
zillow_county = pd.read_csv('../Data/zillow county.csv')
employment_levels = pd.read_csv('../Data/employment_data_2017_2019_combined.csv')
tariffs = pd.read_csv('../Data/ustariffs-by-county.csv')
population = pd.read_csv('../Data/population_data.csv')

# Begin Editing population file
population.rename(columns={'Unnamed: 0': 'County'}, inplace=True)
population = population[['County','2017','2018','2019']]

# Initialize the AddFIPS object
af = addfips.AddFIPS()

def add_fips_code(row):
    county_text = row['County']
    
    if '.' in county_text:
        parts = county_text.split('.')
        if len(parts) > 1:
            county_state = parts[1].strip()
        else:
            county_state = county_text
    else:
        county_state = county_text
    
    if ',' in county_state:
        county, state = county_state.split(',', 1)
        county = county.strip()
        state = state.strip()
    else:
        county = county_state
        state = None
    
    if "County" in county:
        county = county.replace(' County', '')
    
    if state and county:
        fips = af.get_county_fips(county, state=state)
        return fips
    else:
        return None

# Apply the function to create a new FIPS column
population['fips'] = population.apply(add_fips_code, axis=1)

# Reset index and melt the DataFrame
population_reset = population.reset_index(drop=True)
population_long = pd.melt(
    population_reset,
    id_vars=['County', 'fips'],
    value_vars=['2017', '2018', '2019'],
    var_name='year',
    value_name='population'
)

population_long['County'] = population_long['County'].str.replace(r'^\.\s*', '', regex=True)
population_long = population_long.sort_values(['County', 'year'])
population_long = population_long.reset_index(drop=True)
population_long['year'] = population_long['year'].astype(int)

# Clean employment data
columns_to_keep = ['area_fips','year','qtr','area_title','month1_emplvl','month2_emplvl','month3_emplvl']
employment_levels = employment_levels[columns_to_keep]

def create_county_fips(state_fips, municipal_code):
    state_fips_str = str(state_fips).zfill(2)
    municipal_code_str = str(municipal_code).zfill(3)
    county_fips = state_fips_str + municipal_code_str
    return county_fips

# Apply the Function to Create new FIPS column
zillow_county['CountyFIPS'] = zillow_county.apply(
    lambda row: create_county_fips(row['StateCodeFIPS'], row['MunicipalCodeFIPS']), 
    axis=1
)

# Filter for county data only
zillow_county = zillow_county[zillow_county['RegionType'] == 'county']

# Clean up Zillow county Data 
columns_to_drop = ['RegionID','SizeRank', 'RegionType','StateName','State','StateCodeFIPS','MunicipalCodeFIPS','Metro']
zillow_county = zillow_county.drop(columns = columns_to_drop)

# Melt the date columns into rows
melted_df = pd.melt(
    zillow_county,
    id_vars=['RegionName', 'CountyFIPS'],
    var_name='Date',
    value_name='HousingPrice'
)

melted_df['Date'] = pd.to_datetime(melted_df['Date'])
melted_df = melted_df.sort_values(['Date', 'RegionName'])
melted_df = melted_df.reset_index(drop=True)

# Filter to keep only data from 2017-2019
melted_df = melted_df[(melted_df['Date'] >= '2017-01-01') & (melted_df['Date'] <= '2019-12-31')]
melted_df = melted_df.reset_index(drop=True)

# Clean tariffs data frame
tariffs = tariffs[['time','area_fips','tariff']]
tariffs['time'] = pd.to_datetime(tariffs['time'])
tariffs['time'] = tariffs['time'] - timedelta(days=1)
filtered_tariffs = tariffs[(tariffs['time'] >= '2017-01-01') & (tariffs['time'] <= '2019-12-31')]
filtered_tariffs = filtered_tariffs.reset_index(drop=True)
filtered_tariffs['area_fips'] = filtered_tariffs['area_fips'].astype(str)

# Process employment data
county_employment_levels = employment_levels[employment_levels['area_title'].str.contains('County', case=False, na=False)]

transformed_data = []
for _, row in county_employment_levels.iterrows():
    area_fips = row['area_fips']
    area_title = row['area_title']
    year = row['year']
    quarter = row['qtr']
    
    base_month = (quarter - 1) * 3 + 1
    
    for month_idx in range(3):
        month_num = base_month + month_idx
        
        if month_num == 2:
            if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
                day = 29
            else:
                day = 28
        elif month_num in [4, 6, 9, 11]:
            day = 30
        else:
            day = 31
        
        date_str = f"{year}-{month_num:02d}-{day}"
        employment_level = row[f'month{month_idx+1}_emplvl']
        
        transformed_data.append({
            'CountyFIPS': area_fips,
            'RegionName': area_title,
            'Date': date_str,
            'EmploymentLevel': employment_level
        })

transformed_employment_df = pd.DataFrame(transformed_data)
transformed_employment_df['Date'] = pd.to_datetime(transformed_employment_df['Date'])
transformed_employment_df = transformed_employment_df.sort_values(['CountyFIPS', 'Date']).reset_index(drop=True)

# Merge all datasets
merged_df = pd.merge(
    transformed_employment_df,
    melted_df,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['CountyFIPS', 'Date']
)

merged_df = pd.merge(
    merged_df,
    filtered_tariffs,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['area_fips','time']
)

merged_df['year'] = merged_df['Date'].dt.year
merged_df = pd.merge(
    merged_df,
    population_long,
    how='inner',
    left_on=['CountyFIPS','year'],
    right_on=['fips','year']
)

# Clean merged data
merged_df = merged_df.drop(columns=['RegionName_x','RegionName_y','area_fips','year','time','County','fips'])
merged_df = merged_df.dropna()
merged_df = merged_df.reset_index(drop=True)
merged_df['population'] = merged_df['population'].astype(str).str.replace(',', '')
merged_df['population'] = pd.to_numeric(merged_df['population'], errors='coerce')
merged_df['HousingPrice'] = np.log(merged_df['HousingPrice'])

# CREATE REGIONAL CLASSIFICATION
print("Creating regional classification...")

# US Census Bureau regional classification based on state FIPS codes
# State FIPS codes are the first 2 digits of the county FIPS
def get_region_from_fips(county_fips):
    """Map county FIPS to US Census regions"""
    try:
        state_fips = int(str(county_fips)[:2])
    except:
        return None
    
    # Northeast: CT(09), ME(23), MA(25), NH(33), NJ(34), NY(36), PA(42), RI(44), VT(50)
    northeast_states = [9, 23, 25, 33, 34, 36, 42, 44, 50]
    
    # Midwest: IL(17), IN(18), IA(19), KS(20), MI(26), MN(27), MO(29), NE(31), ND(38), OH(39), SD(46), WI(55)
    midwest_states = [17, 18, 19, 20, 26, 27, 29, 31, 38, 39, 46, 55]
    
    # South: AL(01), AR(05), DE(10), FL(12), GA(13), KY(21), LA(22), MD(24), MS(28), NC(37), OK(40), SC(45), TN(47), TX(48), VA(51), WV(54), DC(11)
    south_states = [1, 5, 10, 11, 12, 13, 21, 22, 24, 28, 37, 40, 45, 47, 48, 51, 54]
    
    # West: AK(02), AZ(04), CA(06), CO(08), HI(15), ID(16), MT(30), NV(32), NM(35), OR(41), UT(49), WA(53), WY(56)
    west_states = [2, 4, 6, 8, 15, 16, 30, 32, 35, 41, 49, 53, 56]
    
    if state_fips in northeast_states:
        return 'Northeast'
    elif state_fips in midwest_states:
        return 'Midwest'
    elif state_fips in south_states:
        return 'South'
    elif state_fips in west_states:
        return 'West'
    else:
        return 'Unknown'

# Add region column to merged_df
merged_df['CountyFIPS_int'] = merged_df['CountyFIPS'].astype(int)
merged_df['Region'] = merged_df['CountyFIPS_int'].apply(get_region_from_fips)

# Print regional distribution
print("Regional distribution of counties:")
region_counts = merged_df.groupby('Region')['CountyFIPS'].nunique()
print(region_counts)

# Remove any counties with unknown regions
merged_df = merged_df[merged_df['Region'] != 'Unknown']
merged_df = merged_df.reset_index(drop=True)

# REGRESSION ANALYSIS FUNCTIONS
def create_12month_differences(df):
    """Calculate 12-month differences for all variables"""
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['CountyFIPS', 'Date'])
    
    # Calculate 12-month lagged values by county
    df['HousingPrice_12m'] = df.groupby('CountyFIPS')['HousingPrice'].shift(12)
    df['EmploymentLevel_12m'] = df.groupby('CountyFIPS')['EmploymentLevel'].shift(12)
    df['tariff_12m'] = df.groupby('CountyFIPS')['tariff'].shift(12)
    df['population_12m'] = df.groupby('CountyFIPS')['population'].shift(12)
    
    # Calculate 12-month differences
    df['HousingPrice_diff'] = df['HousingPrice'] - df['HousingPrice_12m']
    
    # Employment: calculate log difference
    df = df[(df['EmploymentLevel'] > 0) & (df['EmploymentLevel_12m'] > 0)]
    df['EmploymentLevel_diff'] = np.log(df['EmploymentLevel']) - np.log(df['EmploymentLevel_12m'])
    
    # Tariff: calculate difference
    df['tariff_diff'] = df['tariff'] - df['tariff_12m']
    
    # Use the earlier year's population as weight
    df['weight_population'] = df['population_12m']
    
    # Drop rows with missing 12-month differences
    df = df.dropna(subset=['HousingPrice_diff', 'EmploymentLevel_diff', 'tariff_diff', 'weight_population'])
    
    # Keep only the columns we need
    result_df = df[['CountyFIPS', 'Date', 'HousingPrice_diff', 'tariff_diff', 
                   'EmploymentLevel_diff', 'weight_population']].copy()
    
    # Rename for consistency
    result_df = result_df.rename(columns={
        'HousingPrice_diff': 'HousingPrice',
        'tariff_diff': 'tariff',
        'EmploymentLevel_diff': 'EmploymentLevel',
        'weight_population': 'population'
    })
    
    return result_df

def create_dummy_variables(df):
    """Create county and date dummy variables for fixed effects"""
    # County dummies
    county_dummies = pd.get_dummies(df['CountyFIPS'], prefix='County')
    merged_data_county_dummies = pd.concat([df, county_dummies], axis=1)
    merged_data_county_dummies = merged_data_county_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Date dummies  
    date_dummies = pd.get_dummies(df['Date'], prefix='Date')
    merged_data_date_dummies = pd.concat([df, date_dummies], axis=1)
    merged_data_date_dummies = merged_data_date_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Combined dummies
    merged_data_large = pd.concat([merged_data_county_dummies, date_dummies], axis=1)
    
    return merged_data_county_dummies, merged_data_date_dummies, merged_data_large

def add_stars(coef, pval):
    """Add significance stars to coefficients"""
    if pval < 0.01:
        return f"{coef:.3f}***"
    elif pval < 0.05:
        return f"{coef:.3f}**"
    elif pval < 0.1:
        return f"{coef:.3f}*"
    else:
        return f"{coef:.3f}"

def run_regressions_for_region(data, region_name):
    """Run the 6 regressions for a specific region"""
    print(f"\nRunning regressions for {region_name}...")
    print(f"Number of observations: {len(data)}")
    print(f"Number of counties: {data['CountyFIPS'].nunique()}")
    
    # Create 12-month differences
    panel_data = create_12month_differences(data)
    
    if len(panel_data) == 0:
        print(f"No data available for {region_name} after creating differences")
        return None
    
    print(f"After creating differences - Observations: {len(panel_data)}, Counties: {panel_data['CountyFIPS'].nunique()}")
    
    # Create dummy variables
    merged_data_county_dummies, merged_data_date_dummies, merged_data_large = create_dummy_variables(panel_data)
    
    try:
        # Regression 1: Simple OLS with just tariff change
        reg_1 = sm.OLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff'])
        ).fit()
        
        # Regression 2: WLS with population weights
        reg_2 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff']),
            weights=panel_data['population']
        ).fit()
        
        # Regression 3: WLS with time dummies (but no employment change)
        reg_3 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()
        
        # Regression 4: WLS with time dummies and employment change
        reg_4 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies),
            weights=panel_data['population']
        ).fit()
        
        # Regression 5: WLS with county and time dummies (but no employment change)
        reg_5 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()
        
        # Regression 6: WLS with county and time dummies and employment change
        reg_6 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large),
            weights=panel_data['population']
        ).fit()
        
        return [reg_1, reg_2, reg_3, reg_4, reg_5, reg_6], panel_data
    
    except Exception as e:
        print(f"Error running regressions for {region_name}: {str(e)}")
        return None, panel_data

def create_regression_table(reg_list, region_name):
    """Create a formatted regression table"""
    if reg_list is None:
        return None
        
    # Create an empty dataframe for the table
    columns = ['(1)', '(2)', '(3)', '(4)', '(5)', '(6)']
    rows = ['Δ Tariff', 'Tariff SE', 'Δ Log Employment', 'Employment SE', 'Fixed Effects:', 'County', 'Time', 'Observations', 'R-squared']
    table = pd.DataFrame(index=rows, columns=columns)
    
    # Initialize all cells to empty string
    for col in columns:
        for row in rows:
            table.loc[row, col] = ""
    
    # Extract coefficient values with stars and standard errors
    # Tariff coefficients
    for i, reg in enumerate(reg_list):
        col = columns[i]
        if 'tariff' in reg.params:
            table.loc['Δ Tariff', col] = add_stars(reg.params['tariff'], reg.pvalues['tariff'])
            table.loc['Tariff SE', col] = f"({reg.bse['tariff']:.3f})"
    
    # Employment coefficients (for regressions that include it)
    employment_regs = [reg_list[3], reg_list[5]]  # reg_4 and reg_6
    employment_cols = [columns[3], columns[5]]
    
    for reg, col in zip(employment_regs, employment_cols):
        if 'EmploymentLevel' in reg.params:
            table.loc['Δ Log Employment', col] = add_stars(reg.params['EmploymentLevel'], reg.pvalues['EmploymentLevel'])
            table.loc['Employment SE', col] = f"({reg.bse['EmploymentLevel']:.3f})"
    
    # Fixed effects indicators
    table.loc['Fixed Effects:'] = ""
    table.loc['County'] = ['N', 'N', 'N', 'N', 'Y', 'Y']
    table.loc['Time'] = ['N', 'N', 'Y', 'Y', 'Y', 'Y']
    
    # Model statistics
    table.loc['Observations'] = [f"{int(reg.nobs)}" for reg in reg_list]
    table.loc['R-squared'] = [f"{reg.rsquared:.3f}" for reg in reg_list]
    
    return table

def create_latex_table(reg_list, region_name):
    """Create a LaTeX formatted regression table"""
    if reg_list is None:
        return None
    
    # Determine population weight indicator for each regression
    pop_weights = ['N', 'Y', 'Y', 'Y', 'Y', 'Y']
    
    latex_content = f"""\\begin{{table}}[htbp]
\\centering
\\caption{{Effect of Tariff Changes on 12-Month Housing Price Changes - {region_name} Region}}
\\label{{tab:tariffs_changes_{region_name.lower()}}}
\\begin{{tabular}}{{lcccccc}}
\\toprule
{{}} &       (1) &      (2) &       (3) &       (4) &       (5) &        (6) \\\\
\\midrule
$\\Delta$ Tariff  """
    
    # Add tariff coefficients
    for i, reg in enumerate(reg_list):
        if 'tariff' in reg.params:
            coef_str = add_stars(reg.params['tariff'], reg.pvalues['tariff'])
            if i == 0:
                latex_content += f"&   {coef_str}"
            else:
                latex_content += f" &    {coef_str}"
        else:
            latex_content += " &          "
    
    latex_content += " \\\\\n                 "
    
    # Add tariff standard errors
    for i, reg in enumerate(reg_list):
        if 'tariff' in reg.params:
            se_str = f"({reg.bse['tariff']:.3f})"
            if i == 0:
                latex_content += f"&    {se_str}"
            else:
                latex_content += f" &   {se_str}"
        else:
            latex_content += " &          "
    
    latex_content += " \\\\\n$\\Delta$ Log Employment "
    
    # Add employment coefficients (only for columns 4 and 6)
    for i, reg in enumerate(reg_list):
        if i in [3, 5] and 'EmploymentLevel' in reg.params:  # columns 4 and 6
            coef_str = add_stars(reg.params['EmploymentLevel'], reg.pvalues['EmploymentLevel'])
            latex_content += f" &  {coef_str}"
        else:
            latex_content += " &          "
    
    latex_content += " \\\\\n                 "
    
    # Add employment standard errors
    for i, reg in enumerate(reg_list):
        if i in [3, 5] and 'EmploymentLevel' in reg.params:  # columns 4 and 6
            se_str = f"({reg.bse['EmploymentLevel']:.3f})"
            latex_content += f" &   {se_str}"
        else:
            latex_content += " &          "
    
    latex_content += " \\\\\nFixed Effects:   "
    
    # Add empty row for Fixed Effects label
    for i in range(6):
        latex_content += " &          "
    
    latex_content += " \\\\\nCounty           "
    
    # Add county fixed effects indicators
    county_fe = ['N', 'N', 'N', 'N', 'Y', 'Y']
    for i, indicator in enumerate(county_fe):
        if i == 0:
            latex_content += f"&         {indicator}"
        else:
            latex_content += f" &        {indicator}"
    
    latex_content += " \\\\\nTime             "
    
    # Add time fixed effects indicators
    time_fe = ['N', 'N', 'Y', 'Y', 'Y', 'Y']
    for i, indicator in enumerate(time_fe):
        if i == 0:
            latex_content += f"&        {indicator}"
        else:
            latex_content += f" &         {indicator}"
    
    latex_content += " \\\\\nPopulation Weight "
    
    # Add population weight indicators
    for i, indicator in enumerate(pop_weights):
        if i == 0:
            latex_content += f"&        {indicator}"
        else:
            latex_content += f" &         {indicator}"
    
    latex_content += " \\\\\n\\midrule\nObservations     "
    
    # Add observations
    for i, reg in enumerate(reg_list):
        obs_str = f"{int(reg.nobs)}"
        if i == 0:
            latex_content += f"&     {obs_str}"
        else:
            if len(obs_str) == 4:  # 4-digit number
                latex_content += f" &     {obs_str}"
            else:  # 5-digit number
                latex_content += f" &      {obs_str}"
    
    latex_content += " \\\\\nR-squared        "
    
    # Add R-squared
    for i, reg in enumerate(reg_list):
        rsq_str = f"{reg.rsquared:.3f}"
        if i == 0:
            latex_content += f"&     {rsq_str}"
        else:
            latex_content += f" &     {rsq_str}"
    
    latex_content += """ \\\\
\\bottomrule
\\end{tabular}
\\end{table}
\\FloatBarrier"""
    
    return latex_content

# RUN REGRESSIONS FOR EACH REGION
regional_results = {}
regions = ['Northeast', 'Midwest', 'South', 'West']

for region in regions:
    print(f"\n{'='*60}")
    print(f"PROCESSING {region.upper()} REGION")
    print(f"{'='*60}")
    
    # Filter merged_df to only include counties in this region
    region_data = merged_df[merged_df['Region'] == region].copy()
    
    if len(region_data) == 0:
        print(f"No data available for {region} region")
        continue
    
    # Run regressions for this region
    reg_results, panel_data = run_regressions_for_region(region_data, region)
    
    if reg_results is not None:
        # Create and display table
        table = create_regression_table(reg_results, region)
        
        if table is not None:
            print(f"\nRegression Results for {region} Region:")
            print(table)
            
            # Create LaTeX table
            latex_table = create_latex_table(reg_results, region)
            
            # Store results
            regional_results[region] = {
                'regressions': reg_results,
                'table': table,
                'latex_table': latex_table,
                'panel_data': panel_data
            }
            
            # Print key statistics
            print(f"\nKey Statistics for {region} Region:")
            print(f"Average 12-month tariff change: {panel_data['tariff'].mean():.4f}")
            print(f"Average 12-month log price change: {panel_data['HousingPrice'].mean():.4f}")
            print(f"Tariff coefficient (full model): {reg_results[5].params['tariff']:.4f}")
            print(f"P-value (full model): {reg_results[5].pvalues['tariff']:.4f}")

# SUMMARY COMPARISON
print(f"\n{'='*80}")
print("SUMMARY COMPARISON ACROSS REGIONS")
print(f"{'='*80}")

summary_data = []
for region, results in regional_results.items():
    if results is not None:
        full_model = results['regressions'][5]  # The most complete model
        panel_data = results['panel_data']
        
        summary_data.append({
            'Region': region,
            'Counties': panel_data['CountyFIPS'].nunique(),
            'Observations': len(panel_data),
            'Avg Tariff Change': panel_data['tariff'].mean(),
            'Avg Price Change': panel_data['HousingPrice'].mean(),
            'Tariff Coefficient': full_model.params['tariff'],
            'P-value': full_model.pvalues['tariff'],
            'R-squared': full_model.rsquared
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(4))

# Save results
print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

# Create LaTeX_Tables directory if it doesn't exist
os.makedirs('LaTeX_Tables', exist_ok=True)

# Save individual tables
for region, results in regional_results.items():
    if results is not None:
        # Save LaTeX table only
        tex_filename = f"regression_table_{region.lower()}_region.tex"
        with open(f"../LaTeX_Tables/{tex_filename}", 'w') as f:
            f.write(results['latex_table'])
        print(f"Saved {tex_filename}")

# Save summary table
summary_df.to_csv("../LaTeX_Tables/regional_summary_comparison.csv", index=False)
print("Saved regional_summary_comparison.csv")

print(f"\n{'='*60}")
print("ANALYSIS COMPLETE")
print(f"{'='*60}")

Creating regional classification...
Regional distribution of counties:
Region
Midwest       984
Northeast     208
South        1124
West          266
Name: CountyFIPS, dtype: int64

PROCESSING NORTHEAST REGION

Running regressions for Northeast...
Number of observations: 7488
Number of counties: 208
After creating differences - Observations: 4992, Counties: 208

Regression Results for Northeast Region:
                        (1)        (2)      (3)       (4)       (5)       (6)
Δ Tariff          -0.004***  -0.006***  -0.001*  0.002***  0.008***  0.008***
Tariff SE           (0.000)    (0.001)  (0.001)   (0.001)   (0.001)   (0.001)
Δ Log Employment                                 0.338***               0.033
Employment SE                                     (0.020)             (0.025)
Fixed Effects:                                                               
County                    N          N        N         N         Y         Y
Time                      N          N        Y 